# Sber GigaChat LLM Run for Strategy Payloads

Этот ноутбук нужен только для LLM-прогона уже готовых payloads.

Он НЕ пересобирает цепочки, НЕ считает признаки и НЕ стандартизирует данные заново.

Вход:

- `outputs/chain_grouping_llm_model_experiments/llm_payloads_speak_set_*_summary.json`
- `llm_prompts/final_commentary_system_prompt_v3.txt`

Выход:

- raw/parsed ответы модели в JSONL;
- CSV с комментариями;
- TXT-файл вида `[t_start -> t_end] commentary`, чтобы удобно читать глазами.

По умолчанию API выключен: `RUN_API = False`.


## 1. Импорты и конфигурация

Здесь задаём модель, стратегию группировки и лимиты вызовов.

Для первого теста лучше поставить:

```python
STRATEGIES_TO_RUN = ['related_stop_cap']
MAX_PAYLOADS_PER_STRATEGY = 5
RUN_API = True
```

Когда убедишься, что формат хороший, можно увеличить лимит или прогнать все стратегии.


In [ ]:
from pathlib import Path
import sys
import os
import json
import re
import time
import hashlib
from datetime import datetime

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from statsbomb_toolkit.gigachat import GigaChatClient

# --- Paths ---
PAYLOAD_DIR = ROOT / 'outputs' / 'chain_grouping_llm_model_experiments'
PROMPT_PATH = ROOT / 'llm_prompts' / 'final_commentary_system_prompt_v3.txt'

RUN_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_DIR = ROOT / 'outputs' / 'gigachat_strategy_runs' / RUN_STAMP
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = ROOT / 'outputs' / 'gigachat_strategy_runs' / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# --- Model ---
MODEL_NAME = 'GigaChat-2-Pro'
TEMPERATURE = 0.0

# --- What to run ---
ALL_STRATEGIES = [
    'event_only',
    'related_only',
    'related_cap',
    'related_stop_cap',
    'time_window',
    'possession_cap',
]

# Для первого теста лучше оставить одну стратегию.
STRATEGIES_TO_RUN = ['related_stop_cap']
# STRATEGIES_TO_RUN = ALL_STRATEGIES

SB360_MODE = 'summary'
MAX_PAYLOADS_PER_STRATEGY = 10  # None -> все payloads выбранной стратегии
MAX_CALLS_TOTAL = 30            # safety limit

RUN_API = False                 # Поставь True, когда готова реально вызывать API
USE_CACHE = True
FORCE_API_CALL = False

print('PAYLOAD_DIR:', PAYLOAD_DIR)
print('PROMPT_PATH:', PROMPT_PATH)
print('OUT_DIR:', OUT_DIR)


## 2. Загрузка системного промпта и payloads

Системный промпт один и тот же для всех стратегий. Это важно для честного сравнения: меняется только способ группировки событий, а инструкция модели остаётся одинаковой.


In [ ]:
SYSTEM_PROMPT = PROMPT_PATH.read_text(encoding='utf-8')
print('system prompt chars:', len(SYSTEM_PROMPT))
print(SYSTEM_PROMPT[:1200])


def load_payloads_for_strategy(strategy: str, sb360_mode: str = 'summary'):
    path = PAYLOAD_DIR / f'llm_payloads_speak_set_{strategy}_{sb360_mode}.json'
    if not path.exists():
        raise FileNotFoundError(f'Payload file not found: {path}')
    payloads = json.loads(path.read_text(encoding='utf-8'))
    return path, payloads

payloads_by_strategy = {}
load_rows = []
for strategy in STRATEGIES_TO_RUN:
    path, payloads = load_payloads_for_strategy(strategy, SB360_MODE)
    if MAX_PAYLOADS_PER_STRATEGY is not None:
        payloads = payloads[:MAX_PAYLOADS_PER_STRATEGY]
    payloads_by_strategy[strategy] = payloads
    load_rows.append({
        'strategy': strategy,
        'payloads_loaded': len(payloads),
        'path': str(path),
    })

load_df = pd.DataFrame(load_rows)
display(load_df)
print('total payloads selected:', sum(len(v) for v in payloads_by_strategy.values()))


## 3. Проверка payload перед API

Смотрим, что именно уйдёт в модель: `selection`, `chain_event_ids`, первые события. Это полезно перед дорогим прогоном.


In [ ]:
def preview_payload(payload, max_events=3):
    return {
        'match_id': payload.get('match_id'),
        'strategy': payload.get('strategy'),
        'selection': payload.get('selection'),
        'chain_event_ids_n': len(payload.get('chain_event_ids') or []),
        'events_n': len(payload.get('events') or []),
        'events_preview': [
            {
                'event_id': ((e.get('event_json') or {}).get('id')),
                'timestamp': ((e.get('event_json') or {}).get('timestamp')),
                'type': (((e.get('event_json') or {}).get('type') or {}).get('name')),
                'team': (((e.get('event_json') or {}).get('team') or {}).get('name')),
                'player': (((e.get('event_json') or {}).get('player') or {}).get('name')),
                'derived_zones': (e.get('derived') or {}).get('zones'),
                'episode_signals': (e.get('derived') or {}).get('episode_signals'),
                'has_360_summary': e.get('sb360_summary') is not None,
            }
            for e in (payload.get('events') or [])[:max_events]
        ]
    }

for strategy, payloads in payloads_by_strategy.items():
    print('\n===', strategy, '===')
    if payloads:
        print(json.dumps(preview_payload(payloads[0]), ensure_ascii=False, indent=2)[:5000])


## 4. Служебные функции: prompt, JSON parsing, cache

Ответ модели должен быть строгим JSON. На всякий случай есть `safe_extract_json`, который пытается вытащить JSON из ответа, если модель добавила лишний текст.


In [ ]:
def make_user_prompt(payload: dict) -> str:
    return 'ВХОДНОЙ JSON EPISODE PAYLOAD:\n' + json.dumps(payload, ensure_ascii=False, indent=2)


def safe_extract_json(text: str):
    if not text:
        return {}
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r'\{.*\}', text, flags=re.DOTALL)
    if not m:
        return {}
    try:
        return json.loads(m.group(0))
    except Exception:
        return {}


def cache_key(model: str, system_prompt: str, user_prompt: str, temperature):
    obj = {
        'model': model,
        'system_prompt': system_prompt,
        'user_prompt': user_prompt,
        'temperature': temperature,
    }
    return hashlib.sha256(json.dumps(obj, ensure_ascii=False, sort_keys=True).encode('utf-8')).hexdigest()


def cache_read(key: str):
    path = CACHE_DIR / f'{key}.json'
    if path.exists():
        return json.loads(path.read_text(encoding='utf-8'))
    return None


def cache_write(key: str, value: dict):
    path = CACHE_DIR / f'{key}.json'
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')


def count_words_ru(text: str) -> int:
    return len(re.findall(r'[A-Za-zА-Яа-яЁё]+', text or ''))


def validate_parsed(parsed: dict, payload: dict):
    events = payload.get('events') or []
    expected_start = ((events[0].get('event_json') or {}).get('timestamp')) if events else ''
    expected_end = ((events[-1].get('event_json') or {}).get('timestamp')) if events else ''
    event_ids = {((e.get('event_json') or {}).get('id')) for e in events}
    event_types = {(((e.get('event_json') or {}).get('type') or {}).get('name')) for e in events}

    commentary = parsed.get('commentary', '') if isinstance(parsed, dict) else ''
    commented_ids = parsed.get('commented_event_ids', []) if isinstance(parsed, dict) else []
    commented_types = parsed.get('commented_event_types', []) if isinstance(parsed, dict) else []

    return {
        'json_valid': int(isinstance(parsed, dict) and bool(parsed)),
        'required_fields': int(isinstance(parsed, dict) and all(k in parsed for k in ['t_start', 't_end', 'commentary'])),
        'time_bounds_match': int(
            isinstance(parsed, dict)
            and str(parsed.get('t_start')) == str(expected_start)
            and str(parsed.get('t_end')) == str(expected_end)
        ),
        'commentary_nonempty': int(bool(str(commentary).strip())),
        'word_count': count_words_ru(commentary),
        'commented_ids_valid': int(all(x in event_ids for x in commented_ids)) if isinstance(commented_ids, list) else 0,
        'commented_types_valid': int(all(x in event_types for x in commented_types)) if isinstance(commented_types, list) else 0,
        'forbidden_temporal_words': int(bool(re.search(r'\b(затем|потом|после этого|далее|в итоге|следом|сначала)\b', str(commentary).lower()))),
    }


## 5. Smoke test: один payload без массового запуска

Эта ячейка делает один API-вызов, если `RUN_API=True`. Используй её, чтобы проверить, что ключи и формат ответа работают.

Нужна переменная окружения с ключом Sber/GigaChat. В твоём клиенте обычно используется `GIGACHAT_BASIC_AUTH` или настройки из `.env`.


In [ ]:
def call_gigachat(client, payload: dict):
    user_prompt = make_user_prompt(payload)
    resp = client.chat_completion(
        prompt=user_prompt,
        system_prompt=SYSTEM_PROMPT,
        model=MODEL_NAME,
        temperature=TEMPERATURE,
    )
    raw = ((resp.get('choices') or [{}])[0].get('message') or {}).get('content', '')
    parsed = safe_extract_json(raw)
    usage = resp.get('usage', {}) if isinstance(resp, dict) else {}
    return raw, parsed, usage, resp

first_strategy = STRATEGIES_TO_RUN[0]
first_payload = payloads_by_strategy[first_strategy][0]
print('smoke strategy:', first_strategy)
print('smoke selection:', first_payload.get('selection'))

if RUN_API:
    client = GigaChatClient(model=MODEL_NAME)
    raw, parsed, usage, resp = call_gigachat(client, first_payload)
    print('RAW:', raw[:2000])
    print('PARSED:', json.dumps(parsed, ensure_ascii=False, indent=2))
    print('USAGE:', usage)
    print('CHECKS:', validate_parsed(parsed, first_payload))
else:
    print('RUN_API=False, smoke API call skipped.')


## 6. Массовый прогон выбранных стратегий

Эта ячейка сохраняет результат построчно в JSONL сразу после каждого ответа. Если остановить ноутбук, уже полученные ответы не потеряются.

Файлы на выходе:

- `gigachat_outputs.jsonl` — полный лог;
- `gigachat_outputs.csv` — таблица;
- `gigachat_commentaries.txt` — удобно читать комментарии глазами.


In [ ]:
def run_generation():
    client = GigaChatClient(model=MODEL_NAME)
    out_jsonl = OUT_DIR / 'gigachat_outputs.jsonl'
    rows = []
    calls = 0

    with out_jsonl.open('w', encoding='utf-8') as f:
        for strategy in STRATEGIES_TO_RUN:
            for payload_idx, payload in enumerate(payloads_by_strategy[strategy], start=1):
                if MAX_CALLS_TOTAL is not None and calls >= MAX_CALLS_TOTAL:
                    print('MAX_CALLS_TOTAL reached:', MAX_CALLS_TOTAL)
                    return rows

                user_prompt = make_user_prompt(payload)
                ck = cache_key(MODEL_NAME, SYSTEM_PROMPT, user_prompt, TEMPERATURE)

                cached = cache_read(ck) if (USE_CACHE and not FORCE_API_CALL) else None
                cache_hit = int(cached is not None)
                error = None
                raw = ''
                parsed = {}
                usage = {}

                t0 = time.perf_counter()
                if cached is not None:
                    raw = cached.get('raw', '')
                    parsed = cached.get('parsed', {})
                    usage = cached.get('usage', {})
                    error = cached.get('error')
                else:
                    try:
                        raw, parsed, usage, _resp = call_gigachat(client, payload)
                    except Exception as e:
                        error = repr(e)

                    if USE_CACHE:
                        cache_write(ck, {
                            'raw': raw,
                            'parsed': parsed,
                            'usage': usage,
                            'error': error,
                        })

                latency_ms = round((time.perf_counter() - t0) * 1000, 2)
                checks = validate_parsed(parsed, payload)
                row = {
                    'strategy': strategy,
                    'payload_idx': payload_idx,
                    'match_id': payload.get('match_id'),
                    'model': MODEL_NAME,
                    'temperature': TEMPERATURE,
                    'selection': payload.get('selection'),
                    'chain_event_ids': payload.get('chain_event_ids'),
                    'cache_hit': cache_hit,
                    'error': error,
                    'latency_ms': latency_ms,
                    'prompt_tokens': usage.get('prompt_tokens') if isinstance(usage, dict) else None,
                    'completion_tokens': usage.get('completion_tokens') if isinstance(usage, dict) else None,
                    'total_tokens': usage.get('total_tokens') if isinstance(usage, dict) else None,
                    'raw': raw,
                    'parsed': parsed,
                    **checks,
                }
                rows.append(row)
                f.write(json.dumps(row, ensure_ascii=False) + '\n')
                f.flush()

                calls += int(cache_hit == 0)
                if len(rows) % 5 == 0:
                    print('rows:', len(rows), 'api calls:', calls, 'last strategy:', strategy)

    return rows

if RUN_API:
    rows = run_generation()
else:
    rows = []
    print('RUN_API=False, generation skipped.')


## 7. Сохранение CSV и TXT для чтения

Если ты остановила прогон и переменная `rows` пустая, ячейка попробует перечитать `gigachat_outputs.jsonl` из `OUT_DIR`.


In [ ]:
def read_jsonl(path: Path):
    if not path.exists():
        return []
    out = []
    with path.open(encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out

if not rows:
    rows = read_jsonl(OUT_DIR / 'gigachat_outputs.jsonl')

print('rows loaded:', len(rows))

flat_rows = []
for r in rows:
    p = r.get('parsed') if isinstance(r.get('parsed'), dict) else {}
    sel = r.get('selection') if isinstance(r.get('selection'), dict) else {}
    flat_rows.append({
        'strategy': r.get('strategy'),
        'payload_idx': r.get('payload_idx'),
        'match_id': r.get('match_id'),
        'model': r.get('model'),
        'soft_label_3': sel.get('soft_label_3'),
        'commenting_mode': sel.get('commenting_mode'),
        't_start': p.get('t_start'),
        't_end': p.get('t_end'),
        'commentary': p.get('commentary'),
        'commented_event_ids': json.dumps(p.get('commented_event_ids', []), ensure_ascii=False),
        'commented_event_types': json.dumps(p.get('commented_event_types', []), ensure_ascii=False),
        'json_valid': r.get('json_valid'),
        'required_fields': r.get('required_fields'),
        'time_bounds_match': r.get('time_bounds_match'),
        'commentary_nonempty': r.get('commentary_nonempty'),
        'word_count': r.get('word_count'),
        'commented_ids_valid': r.get('commented_ids_valid'),
        'commented_types_valid': r.get('commented_types_valid'),
        'forbidden_temporal_words': r.get('forbidden_temporal_words'),
        'error': r.get('error'),
    })

results_df = pd.DataFrame(flat_rows)
display(results_df.head(30))

csv_path = OUT_DIR / 'gigachat_outputs.csv'
results_df.to_csv(csv_path, index=False)
print('saved csv:', csv_path)

txt_path = OUT_DIR / 'gigachat_commentaries.txt'
with txt_path.open('w', encoding='utf-8') as f:
    for _, row in results_df.iterrows():
        f.write(f"[{row.get('strategy')}] [{row.get('t_start')} -> {row.get('t_end')}] {row.get('commentary')}\n")
print('saved txt:', txt_path)

summary_cols = [
    'json_valid', 'required_fields', 'time_bounds_match', 'commentary_nonempty',
    'commented_ids_valid', 'commented_types_valid', 'forbidden_temporal_words',
]
if len(results_df):
    summary = results_df.groupby('strategy')[summary_cols].mean(numeric_only=True).reset_index()
    display(summary)
    summary_path = OUT_DIR / 'gigachat_metrics_by_strategy.csv'
    summary.to_csv(summary_path, index=False)
    print('saved summary:', summary_path)
